In [ ]:
import glob
import os

import h5py
import numpy as np
import matplotlib.pyplot as plt
import dustpy.constants as c

In [ ]:
# load the snapshot of the time you want to plot
# please note, that the folders runs/a* have to contain the respective files for the alpha-values you want to compare
files = sorted(glob.glob('runs/*/data0010.hdf5'))

alpha_map = {
    'ad': 0.0,      # DustPy-ref
    'a2': 1e-2,
    'a3': 1e-3,
    'a4': 1e-4,
    'a5': 1e-5,
}

runs = {}

for file in files:
    folder = os.path.basename(os.path.dirname(file))

    with h5py.File(file, 'r') as f:
        r = f['grid/r'][:] / c.au
        T = f['gas/T'][:]

    runs[alpha_map[folder]] = (r, T)

# Reference = smallest alpha > 0
alphas = sorted([a for a in runs.keys() if a > 0])

r_ref, T_ref = runs[min(alphas)]

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(13,5),
    dpi=300,
    constrained_layout=True
)

# colorscale
colors = plt.cm.viridis(np.linspace(0, 0.85, len(alphas)))

if 0.0 in runs:

    r, T = runs[0.0]

    ax1.loglog(r, T, '--', color='grey',  lw=2.5, label='DustPy')

    ax2.semilogx(r, T / T_ref,'--', color="k", lw=2.5, label="DustPy")

for color, alpha in zip(colors, alphas):

    r, T = runs[alpha]

    ax1.loglog(r, T, color=color, lw=2, label=rf"$\alpha={alpha:.0e}$")

    ax2.semilogx(r, T / T_ref, color=color,lw=2)

ax1.set_xlabel('radius [au]')
ax1.set_ylabel('temperature [K]')
ax1.set_xlim(1,900)
ax1.grid(True, which="both", alpha=0.3)

ax2.set_xlabel('radius [au]')
ax2.set_ylabel(r'$T/T_{\alpha=10^{-5}}$')
ax2.set_xlim(1,900)
ax2.grid(True, which='both', alpha=0.3)

ax2.axhline(1.0, color='0.5', ls=":", lw=1)

ax1.legend(loc = 'best')

plt.show()